In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from torchvision import models
import ssl

In [35]:
# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [36]:
# 1. Tiền xử lý ảnh
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize về 224x224
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Chuẩn hóa theo ImageNet
])

In [37]:
# 2. Load dữ liệu từ thư mục (đã chia train/valid/test)
data_dir = "data/cic_ddos_2019_images"
batch_size = 32

In [38]:
train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
valid_dataset = datasets.ImageFolder(root=f"{data_dir}/valid", transform=transform)
test_dataset = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

In [ ]:
# Sử dụng khi thuê GPU trên máy ảo để tải base model
ssl._create_default_https_context = ssl._create_unverified_context

In [39]:
# 3. Load mô hình EfficientNet từ torchvision
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
num_ftrs = model.classifier[1].in_features  # Số input của layer cuối
model.classifier[1] = nn.Linear(num_ftrs, len(train_dataset.classes))  # Số class = số thư mục nhãn


In [40]:
model = model.to(device)

In [41]:
# 4. Cấu hình optimizer, loss function
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [42]:
# 5. Hàm đánh giá mô hình
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

In [ ]:
# 6. Huấn luyện mô hình
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_acc = 100 * correct / total
    val_acc = evaluate(model, valid_loader, device)

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%")

In [ ]:
# 7. Lưu mô hình
torch.save(model.state_dict(), "models/cic-ddos2019/efficientnet.pth")
print("Training complete! Model saved as efficientnet_ddos.pth.")

# 8. Kiểm tra mô hình trên tập test
test_acc = evaluate(model, test_loader, device)
print(f"Test Accuracy: {test_acc:.2f}%")